In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv
from sklearn.metrics import r2_score
import numpy as np
import pickle
import random
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast
# Keep dependency warnings visible during reproduction; do not silence them globally.

# Each student run uses one evaluation seed for initialization and batch order.
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Teacher architecture compatible with both the original SimpleGCN checkpoints
# and GCNTeacher checkpoints after key normalization below.
class SimpleGCN(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h
        self.dropout = nn.Dropout(dropout)
        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        else:
            self.final_dim = hidden_dims[-1]
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        if hasattr(self, 'edge_norm') and data.edge_attr is not None:
            _ = self.edge_norm(data.edge_attr)
        u = getattr(data, 'u', None)
        if hasattr(self, 'global_norm') and u is not None:
            u = self.global_norm(u)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)
        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze(-1)
        return (out, h) if return_feat else out

class EnhancedGCN(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None
        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h
        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim//2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        if self.edge_norm and hasattr(data, 'edge_attr'):
            _ = self.edge_norm(data.edge_attr)
        u = getattr(data, 'u', None)
        if u is not None:
            u = self.global_norm(u)
            gf = self.global_mlp(u)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)
        pooled = global_mean_pool(x, data.batch)
        h = torch.cat([pooled, gf], dim=1) if u is not None else pooled
        out = self.output_mlp(h).squeeze(-1)
        return (out, h) if return_feat else out

class GateNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_teachers):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_teachers)
    def forward(self, h):
        a = F.relu(self.fc1(h))
        return F.softmax(self.fc2(a), dim=-1)

class Adapter(nn.Module):
    def __init__(self, dim_s, dim_t):
        super().__init__()
        self.linear = nn.Linear(dim_s, dim_t)
    def forward(self, h):
        return self.linear(h)

def create_data_loader(graph_list, batch_size=32, shuffle=True):
    data_list = []
    for g in graph_list:
        data_list.append(Data(
            x=g['x'], edge_index=g['edge_index'],
            edge_attr=g.get('edge_attr', None), u=g.get('u', None),
            y=g['y'], y_soft=(g['y_soft'].reshape(1, -1)
                               if g.get('y_soft') is not None else None)
        ))
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

def normalize_teacher_state_dict(state_dict):
    """Translate the original GCNTeacher layer names to the student loader names."""
    if any(k.startswith('norm.') or k.startswith('output.') for k in state_dict):
        return {
            ('node_norm.' + key[len('norm.'):]) if key.startswith('norm.')
            else ('output_mlp.' + key[len('output.'):]) if key.startswith('output.')
            else key: value
            for key, value in state_dict.items()
        }
    return state_dict

def train_model(
    train_dir, val_dir, teacher_paths, save_path,
    gate_hidden=128, hint_lambda=5.0, weight_ratio=(0.6,0.4),
    hidden_dims=[128,128], dropout=0.1,
    epochs=500, batch_size=64, lr=1e-3, min_lr=1e-4,
    lr_patience=20, es_patience=50,
    seed=42
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_seed(seed)
    train_path = os.path.join(train_dir, 'graph_data.pt')
    val_path = os.path.join(val_dir, 'graph_data.pt')
    for graph_path in (train_path, val_path):
        if not os.path.isfile(graph_path):
            raise FileNotFoundError(f"Required graph data not found: {graph_path}")
    # Load only trusted graph files created by the feature extraction pipeline.
    train_g = torch.load(train_path, weights_only=False)
    val_g = torch.load(val_path, weights_only=False)
    if len(teacher_paths) != 5 or any(not os.path.isfile(p) for p in teacher_paths):
        raise FileNotFoundError("Five existing teacher checkpoint paths are required.")
    # Missing soft labels must not be replaced with the ground-truth targets.
    for split_name, graphs in (("training", train_g), ("validation", val_g)):
        for sample_idx, graph in enumerate(graphs):
            soft = graph.get('y_soft')
            if not isinstance(soft, torch.Tensor) or soft.numel() != len(teacher_paths):
                raise ValueError(
                    f"{split_name} graph {sample_idx} must contain y_soft with "
                    f"{len(teacher_paths)} aligned teacher predictions."
                )

    # Estimate normalization parameters from training targets only.
    ys = torch.stack([g['y'] for g in train_g]).view(-1)
    y_mean, y_std = ys.mean().item(), ys.std().item()+1e-8
    for g in train_g:
        g['y'] = (g['y'] - y_mean) / y_std
        # y_soft was checked above and is already supplied by the data pipeline.
    for g in val_g:
        g['y'] = (g['y'] - y_mean) / y_std
        # Keep teacher labels unchanged; their scale must match student targets.

    tr_loader = create_data_loader(train_g, batch_size, True)
    va_loader = create_data_loader(val_g, batch_size, False)
    sample = train_g[0]
    n_dim = sample['x'].size(1)
    e_dim = sample.get('edge_attr', None).size(1) if sample.get('edge_attr') is not None else 0
    g_dim = sample.get('u', None).size(1) if sample.get('u') is not None else 0
    student = EnhancedGCN(n_dim, e_dim, g_dim, hidden_dims, dropout).to(device)
    teachers = []
    for p in teacher_paths:
        # Checkpoints are trusted local files produced by teacher training.
        ck = torch.load(p, map_location=device, weights_only=False)
        t = SimpleGCN(ck['node_dim'], ck.get('edge_dim', 0),
                      ck.get('global_dim', 0), ck['hidden_dims'], ck['dropout']).to(device)
        state_dict = normalize_teacher_state_dict(ck['model_state_dict'])
        # Strict loading prevents silently using randomly initialized teacher layers.
        t.load_state_dict(state_dict, strict=True)
        t.requires_grad_(False)
        t.eval()
        teachers.append(t)
    print(f"Loaded {len(teachers)} teachers.")

    K = len(teachers)
    gate = GateNet(student.final_dim, gate_hidden, K).to(device)
    adapter = Adapter(student.final_dim, student.final_dim).to(device)
    optimizer = optim.Adam(list(student.parameters()) + list(gate.parameters()) + list(adapter.parameters()), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=lr_patience, min_lr=min_lr)
    scaler = GradScaler(enabled=(device.type=='cuda'))
    best_r2, patience = -1e9, 0
    history = {'loss': [], 'train_r2': [], 'val_r2': []}

    def eval_loader(loader):
        student.eval()
        ys, ps = [], []
        with torch.no_grad():
            for b in loader:
                b = b.to(device)
                out, _ = student(b, return_feat=True)
                ys.append(b.y.view(-1).cpu().numpy())
                ps.append(out.cpu().numpy())
        return r2_score(np.concatenate(ys), np.concatenate(ps))

    for epoch in range(1, epochs + 1):
        student.train()
        total_loss = 0
        for batch in tr_loader:
            batch = batch.to(device)
            with autocast(enabled=(device.type == 'cuda')):
                pred_s, h_s = student(batch, return_feat=True)
                # Fixed teacher representations; gradient updates apply only to student/gate/adapter.
                with torch.no_grad():
                    Ht = torch.stack([t(batch, return_feat=True)[1] for t in teachers], dim=1)
                w = gate(h_s)
                Ht_g = (w.unsqueeze(-1) * Ht).sum(dim=1)
                loss_hint = F.mse_loss(adapter(h_s), Ht_g)
                fused = (w * batch.y_soft).sum(dim=1)
                pred_f = weight_ratio[0] * pred_s + weight_ratio[1] * fused
                loss = (weight_ratio[0] * F.mse_loss(pred_f, batch.y.view(-1)) +
                        weight_ratio[1] * F.mse_loss(pred_s, fused) +
                        hint_lambda * loss_hint)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        train_r2 = eval_loader(tr_loader)
        val_r2 = eval_loader(va_loader)
        avg_loss = total_loss / len(tr_loader)
        lr_now = optimizer.param_groups[0]['lr']
        history['loss'].append(avg_loss)
        history['train_r2'].append(train_r2)
        history['val_r2'].append(val_r2)

        if epoch % 30 == 0:
            print(f"Epoch {epoch} | Loss {avg_loss:.4f} | Train R2 {train_r2:.4f} | Val R2 {val_r2:.4f} | LR {lr_now:.1e}")

        scheduler.step(avg_loss)

        if val_r2 > best_r2:
            best_r2, patience = val_r2, 0
            torch.save({
                'model_state_dict': student.state_dict(),
                'gate_state': gate.state_dict(),
                'adapter_state': adapter.state_dict(),
                'y_mean': y_mean,
                'y_std': y_std,
                'history': history,
                'node_dim': n_dim,
                'edge_dim': e_dim,
                'global_dim': g_dim,
                'hidden_dims': hidden_dims,
                'dropout': dropout
            }, save_path)
            if epoch % 30 != 0:
                print(f"Saved best model at Epoch {epoch}")
        else:
            patience += 1
            if patience >= es_patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    print(f"Training complete, best Val R2={best_r2:.4f}")

    return best_r2, save_path

if __name__ == '__main__':
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    # This notebook performs grid search; these values are candidates, not
    # per-seed hyperparameters reported as optimized in the manuscript.
    hint_lambdas = [1, 5, 10, 20]
    weight_ratios = [(0.4, 0.6), (0.5,0.5), (0.6, 0.4), (0.7,0.3)]
    gate_hiddens = [64, 128, 256]

    # Place training/validation graph_data.pt under these directories.
    train_dir = os.path.join("data-set", "train")
    val_dir = os.path.join("data-set", "validation")
    # Keep teacher order aligned with the five columns of each graph's y_soft.
    teacher_paths = [
        os.path.join("checkpoints", "teachers", f"{name}.pt")
        for name in ("qcut", "elem", "molwt", "fp", "scaffold")
    ]
    save_root = os.path.join("checkpoints", "students")
    os.makedirs(save_root, exist_ok=True)

    epochs = 1000; batch_size = 64; lr = 1e-3; min_lr = 5e-5
    lr_patience = 30; es_patience = 100
    hidden_dims = [128,128]; dropout = 0.1

    results = []

    for hint_lambda in hint_lambdas:
        for weight_ratio in weight_ratios:
            for gate_hidden in gate_hiddens:
                combo_key = f"hl{hint_lambda}_wr{weight_ratio[0]}_{weight_ratio[1]}_gh{gate_hidden}"
                r2s = []
                print(f"\n=== Combo: {combo_key} ===")
                for seed in seeds:
                    print(f"-- Seed {seed}")
                    set_seed(seed)
                    save_path = os.path.join(save_root, f"student_{combo_key}_seed{seed}.pt")
                    best_r2, _ = train_model(
                        train_dir=train_dir,
                        val_dir=val_dir,
                        teacher_paths=teacher_paths,
                        save_path=save_path,
                        gate_hidden=gate_hidden,
                        hint_lambda=hint_lambda,
                        weight_ratio=weight_ratio,
                        hidden_dims=hidden_dims,
                        dropout=dropout,
                        epochs=epochs,
                        batch_size=batch_size,
                        lr=lr,
                        min_lr=min_lr,
                        lr_patience=lr_patience,
                        es_patience=es_patience,
                        seed=seed,
                    )
                    r2s.append(best_r2)
                mean_r2 = np.mean(r2s)
                std_r2 = np.std(r2s)
                print(f"Combo {combo_key}: R2s={r2s}, Mean={mean_r2:.4f}, Std={std_r2:.4f}\n")
                results.append({
                    'combo': combo_key,
                    'r2_list': r2s,
                    'r2_mean': mean_r2,
                    'r2_std': std_r2
                })

    with open(os.path.join(save_root, 'results_summary.pkl'), 'wb') as f:
        pickle.dump(results, f)
    print("All experiments done. Summary saved to results_summary.pkl")

    print("Grid search results are stored as validation summaries only.")
